순서: `ner_documents -> ner_rule_results -> ner_runs -> ner_reviews`

In [39]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

#--DATA_VERSION: 적재 대상 데이터 버전 수정 필요------------------------------------------- 
DATA_VERSION = 'all_pass_ner_tagged_5'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / DATA_VERSION
RULE_INPUT_DIR = PROJECT_ROOT / 'data' / 'filter' / 'rule_detect_data' / DATA_VERSION

#--DOCUMENT_ROOT: 적재 대상 데이터 경로 수정 필요------------------------------------------- 
# DOCUMENT_ROOT = PROJECT_ROOT / 'data' / 'test' / 'test_data_v2'
# DOCUMENT_FILES = sorted(DOCUMENT_ROOT.glob('*.json'))
DOCUMENT_ROOT = PROJECT_ROOT / 'data' / 'org' / DATA_VERSION
DOCUMENT_FILES = sorted(DOCUMENT_ROOT.glob('*.json'))
DOCUMENT_FILES

#--RUN_PATHS: 적재 대상 데이터 경로 수정 필요----------------------------------------------- 
# RUN_PATHS = sorted(OUTPUT_ROOT.glob('run_20261111_*'))
# RUN_PATHS

[WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/강원일보.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/경기일보.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/경향신문.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/노컷뉴스.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/뉴스1.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/뉴스핌.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/뉴시스.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/디지털타임스.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/마이데일리.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/부산일보.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/서울신문.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/세계일보.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/스포츠경향.json'),
 WindowsPath('c:/work/ner/data/org/all_pass_ner_tagged_5/스포츠투데이.json'),
 WindowsPath('c:/

#### 0. 테이블 초기화 및 생성

재적재할 때만 `RESET_TABLES = True`로 변경합니다.

In [13]:
from src.config import get_postgre_connect
from src.postgresql.insert import ner_documents, ner_rule_results, ner_runs, ner_reviews, ner_corrected_sentences

#--RESET_TABLES: 초기에 테이블 구조가 변경될 시 True로 변경
RESET_TABLES = False

def reset_tables():
    with get_postgre_connect() as connection:
        connection.execute('DROP TABLE IF EXISTS ner_rule_results, ner_reviews, ner_corrected_sentences, ner_documents, ner_runs CASCADE')
    ner_runs.create_table()
    ner_documents.create_table()
    ner_corrected_sentences.create_table()
    ner_reviews.create_table()
    ner_rule_results.create_table()
    print('테이블 초기화 및 생성 완료')

if RESET_TABLES:
    reset_tables()
else:
    print('초기화 생략')

ner_runs 테이블 생성 완료
ner_documents 테이블 생성 완료
ner_corrected_sentences 테이블 생성 완료
ner_reviews 테이블 생성 완료
ner_rule_results 테이블 생성 완료
테이블 초기화 및 생성 완료


##### 1. ner_documents

테스트 예시 파일은 `data/test/test_data_v2/강원일보.json`이며, 실제 적재는 해당 폴더의 JSON 전체를 처리합니다. 운영에서는 `data/org/원천데이터폴더명/*.json`으로 바꿉니다.

In [15]:
# 20분
for document_path in DOCUMENT_FILES:
    count = ner_documents.insert_ner_documents(document_path, data_version=DATA_VERSION)
    print(document_path, count)

c:\work\ner\data\org\all_pass_ner_tagged_4\강원일보.json 163534
c:\work\ner\data\org\all_pass_ner_tagged_4\경기일보.json 245835
c:\work\ner\data\org\all_pass_ner_tagged_4\경향신문.json 573387
c:\work\ner\data\org\all_pass_ner_tagged_4\노컷뉴스.json 718049
c:\work\ner\data\org\all_pass_ner_tagged_4\뉴스1.json 2372701
c:\work\ner\data\org\all_pass_ner_tagged_4\뉴스핌.json 1068376
c:\work\ner\data\org\all_pass_ner_tagged_4\뉴시스.json 1640260
c:\work\ner\data\org\all_pass_ner_tagged_4\디지털타임스.json 483516
c:\work\ner\data\org\all_pass_ner_tagged_4\마이데일리.json 607054
c:\work\ner\data\org\all_pass_ner_tagged_4\부산일보.json 476427
c:\work\ner\data\org\all_pass_ner_tagged_4\서울신문.json 657914
c:\work\ner\data\org\all_pass_ner_tagged_4\세계일보.json 746097
c:\work\ner\data\org\all_pass_ner_tagged_4\스포츠경향.json 562565
c:\work\ner\data\org\all_pass_ner_tagged_4\스포츠투데이.json 249807
c:\work\ner\data\org\all_pass_ner_tagged_4\아시아투데이.json 787219
c:\work\ner\data\org\all_pass_ner_tagged_4\이데일리.json 1607403
c:\work\ner\data\org\all_pass_n

##### 2. ner_rule_results

룰 PKL은 `data_version` 기준으로 한 번만 적재합니다.

In [ ]:
# ner_rule_results.insert_ner_rule_results(RULE_INPUT_DIR, data_version=DATA_VERSION)

3002

##### 2-1. ner_rule_results_2

`run_rule_pipeline_2.ipynb`에서 생성한 `candidate_rule_results.csv`를 원문·개체 정보와 전체 룰/패턴 형태로 적재합니다.

In [ ]:
# 11분
press_list = ["일간투데이","충청일보","강원일보","경기일보","경향신문","노컷뉴스","뉴스1","뉴스핌","뉴시스","디지털타임스","마이데일리","부산일보","서울신문","세계일보","스포츠경향","스포츠투데이","아시아투데이","이데일리","이투데이"]

In [ ]:
from src.postgresql.insert import ner_rule_results_2
import importlib
importlib.reload(ner_rule_results_2)

INSERT_CHUNK_SIZE_2 = 2000  # 적재 속도에 따라 변경
RESET_RULE_RESULTS_2 = False # 스키마 변경 시 True, 일반 재적재 시 False

for press in press_list:
    print(press)

    # 언론사별 적재 폴더 설정 (run_rule_pipeline_2 실행 폴더)
    RULE_INPUT_DIR_2 = sorted((PROJECT_ROOT / 'data' / 'filter' / 'rule_detect_data').glob(f'*_{press}'))[-1]
    DATA_VERSION_2 = RULE_INPUT_DIR_2.name

    if RESET_RULE_RESULTS_2:
        ner_rule_results_2.drop_table()
        ner_rule_results_2.create_table()
    else:
        ner_rule_results_2.create_table()
        ner_rule_results_2.insert_ner_rule_results_2(RULE_INPUT_DIR_2, data_version=DATA_VERSION_2, chunk_size=INSERT_CHUNK_SIZE_2)

##### 2-2. ner_rule_results_3

`all_pass_ner_tagged_5_postprocessed` 폴더의 언론사별 후처리 CSV를 ner_rule_results_3에 적재합니다.

In [ ]:
from src.postgresql.insert import ner_rule_results_3
import importlib
importlib.reload(ner_rule_results_3)

INSERT_CHUNK_SIZE_3 = 2000
DATA_VERSION_3 = None

RULE_INPUT_DIR_3 = PROJECT_ROOT / 'data' / 'filter' / 'rule_detect_data' / 'all_pass_ner_tagged_5_postprocessed'
RULE_INPUT_PATHS_3 = sorted(
    path for path in RULE_INPUT_DIR_3.glob('*.csv')
    if path.name not in {'run_summary.csv', 'postprocess_summary.csv'}
)

len(RULE_INPUT_PATHS_3)

19

In [55]:
RESET_RULE_RESULTS_3 = False

if RESET_RULE_RESULTS_3:
    ner_rule_results_3.drop_table()
    ner_rule_results_3.create_table()
else:
    ner_rule_results_3.create_table()
    for rule_input_path in RULE_INPUT_PATHS_3:
        print(rule_input_path.name)
        
        DATA_VERSION_3 = rule_input_path.stem
        
        ner_rule_results_3.insert_ner_rule_results_3(
            rule_input_path,
            data_version=DATA_VERSION_3,
            chunk_size=INSERT_CHUNK_SIZE_3,
        )

ner_rule_results_3 테이블 생성 완료
20261117_191903_일간투데이.csv
20261117_191903_일간투데이.csv: 5027건 처리
ner_rule_results_3 적재 누계: 5027건
20261117_192337_충청일보.csv
20261117_192337_충청일보.csv: 4627건 처리
ner_rule_results_3 적재 누계: 4627건
20261117_192958_강원일보.csv
20261117_192958_강원일보.csv: 2491건 처리
ner_rule_results_3 적재 누계: 2491건
20261117_193354_경기일보.csv
20261117_193354_경기일보.csv: 3292건 처리
ner_rule_results_3 적재 누계: 3292건
20261117_193858_경향신문.csv
20261117_193858_경향신문.csv: 8768건 처리
ner_rule_results_3 적재 누계: 8768건
20261117_195048_노컷뉴스.csv
20261117_195048_노컷뉴스.csv: 11183건 처리
ner_rule_results_3 적재 누계: 11183건
20261117_200619_뉴스1.csv
20261117_200619_뉴스1.csv: 35862건 처리
ner_rule_results_3 적재 누계: 35862건
20261117_205454_뉴스핌.csv
20261117_205454_뉴스핌.csv: 17692건 처리
ner_rule_results_3 적재 누계: 17692건
20261117_211815_뉴시스.csv
20261117_211815_뉴시스.csv: 26438건 처리
ner_rule_results_3 적재 누계: 26438건
20261117_214532_디지털타임스.csv
20261117_214532_디지털타임스.csv: 6983건 처리
ner_rule_results_3 적재 누계: 6983건
20261117_215416_마이데일리.csv
20261117_215416_마

##### 3. ner_runs

예시: `outputs/.../run_20261111_185228/ner_runs.parquet`

In [8]:
RUN_PATHS

[WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_181549'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_185228'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_195129'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_195746'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_201907'),
 WindowsPath('c:/work/ner/outputs/20261110_175720/run_20261111_202437')]

In [9]:
ner_runs.insert_ner_runs(RUN_PATHS[0] / 'ner_runs.parquet')

1

In [10]:
ner_runs.insert_ner_runs(RUN_PATHS[1] / 'ner_runs.parquet')

1

In [11]:
ner_runs.insert_ner_runs(RUN_PATHS[2] / 'ner_runs.parquet')

1

In [12]:
ner_runs.insert_ner_runs(RUN_PATHS[3] / 'ner_runs.parquet')

1

In [13]:
ner_runs.insert_ner_runs(RUN_PATHS[4] / 'ner_runs.parquet')

1

In [14]:
ner_runs.insert_ner_runs(RUN_PATHS[5] / 'ner_runs.parquet')

1

##### 4. ner_reviews

예시: `run_20261111_185228/reports/강원일보/ner_reviews.parquet`
각 run의 `reports/*/ner_reviews.parquet`를 언론사별로 적재합니다.

In [15]:
def insert_reviews_for_run(run_path):
    review_paths = sorted(run_path.glob('reports/*/ner_reviews.parquet'))
    if not review_paths:
        raise FileNotFoundError(f'리뷰 Parquet가 없습니다: {run_path}')
    total = 0
    for review_path in review_paths:
        total += ner_reviews.insert_ner_reviews(review_path)
        print(review_path, '적재 완료')
    return total

In [16]:
insert_reviews_for_run(RUN_PATHS[0])

c:\work\ner\outputs\20261110_175720\run_20261111_181549\reports\강원일보\ner_reviews.parquet 적재 완료


196

In [17]:
insert_reviews_for_run(RUN_PATHS[1])

c:\work\ner\outputs\20261110_175720\run_20261111_185228\reports\강원일보\ner_reviews.parquet 적재 완료


196

In [19]:
insert_reviews_for_run(RUN_PATHS[2])

c:\work\ner\outputs\20261110_175720\run_20261111_195129\reports\강원일보\ner_reviews.parquet 적재 완료


196

In [20]:
insert_reviews_for_run(RUN_PATHS[3])

c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\경기일보\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\경향신문\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\노컷뉴스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\뉴스1\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\뉴스핌\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\뉴시스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\디지털타임스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\마이데일리\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\부산일보\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_195746\reports\서울신문\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_1

2121

In [21]:
insert_reviews_for_run(RUN_PATHS[4])

c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\경기일보\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\경향신문\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\노컷뉴스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\뉴스1\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\뉴스핌\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\뉴시스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\디지털타임스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\마이데일리\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\부산일보\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_201907\reports\서울신문\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_2

2120

In [22]:
insert_reviews_for_run(RUN_PATHS[5])

c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\경기일보\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\경향신문\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\노컷뉴스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\뉴스1\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\뉴스핌\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\뉴시스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\디지털타임스\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\마이데일리\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\부산일보\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_202437\reports\서울신문\ner_reviews.parquet 적재 완료
c:\work\ner\outputs\20261110_175720\run_20261111_2

2120